Imports

In [1]:
import cv2
import json
import torch
from ultralytics import YOLO

CUDA & Capture Data

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

vid_source='Footage.mp4'
print("Source Video:", vid_source)
cap = cv2.VideoCapture(vid_source)
total_frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

Using device: cuda
Source Video: Footage.mp4


YOLO & BoT-SORT

In [ ]:
model = YOLO("yolo12m-tuned.pt")

results = model.track(source=vid_source, persist=True, save=False, classes=0, tracker="botsort.yaml")

b_boxes = {}
p = {} # Class confidence tracker for DeepFace
high_p_boxes = {} # Highest confidence frames

for frame_number, r in enumerate(results):
    boxes = r.boxes

    for i, box in enumerate(boxes):
        xyxy = box.xyxy[0]  # Bounding box in [x_min, y_min, x_max, y_max]
        track_id = int(box.id[0])
        conf = float(box.conf[0]) # Confidence score

        xyxy_clean = str(xyxy)
        xyxy_clean = xyxy_clean.replace("tensor(", "").replace(")", "").strip("[]")
        values = xyxy_clean.split(",")
        xyxy_clean = [int(float(value.strip())) for value in values]

        if track_id not in b_boxes:
            b_boxes[track_id] = []
        b_boxes[track_id].append(xyxy_clean)

        if track_id not in p or conf > p[track_id]:
            p[track_id] = conf
            high_p_boxes[track_id] = (frame_number, xyxy_clean)

# Bounding box dictionary will have gaps for certain values since sometimes the tracked target is lost
# This will cause issues with running inference on the actions since it occurs frame-by-frame.
# Fills in bounding boxes with proximate values.
for track_id in b_boxes:
    while len(b_boxes[track_id]) < total_frames:
        b_boxes[track_id].append(b_boxes[track_id][-1])


output_data = {
    "total_frames": int(total_frames),
    "b_boxes": b_boxes,
    "high_conf_boxes": high_p_boxes,
    "class_confidences": p,
}

# Convert dictionary keys to strings for JSON compatibility
output_data["b_boxes"] = {str(k): v for k, v in output_data["b_boxes"].items()}
output_data["high_conf_boxes"] = {str(k): v for k, v in output_data["high_conf_boxes"].items()}
output_data["class_confidences"] = {str(k): v for k, v in output_data["class_confidences"].items()}

# Save to JSON file
output_filename = "tracking_output.json"
with open(output_filename, "w") as f:
    json.dump(output_data, f, indent=2)

print(f"Tracking data saved to {output_filename}")